# Analisis Estadistico y Graficas - BIOptimization

Este cuaderno realiza la extraccion y analisis estadistico de las 10 corridas experimentales de optimizacion de hiperparametros de la red RBF mediante el algoritmo SSA, persistidas en MongoDB Atlas.

In [ ]:
import os
import sys

# Agregar la raiz del proyecto al path de Python para que Jupyter encuentre 'src'
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.db import MongoManager

# Inicializar conexion con MongoDB Atlas
db = MongoManager()
print("Estado de conexion con MongoDB:", db.connected)

In [ ]:
# Extraer las ultimas 10 corridas del experimento ssa_rbf
if db.connected:
    cursor = db.collection.find({"algorithm": "ssa_rbf"}).sort("timestamp", -1).limit(10)
    runs = list(cursor)
    print(f"Se extrajeron {len(runs)} corridas de MongoDB Atlas.")
else:
    print("Modo Offline: No hay conexion activa a MongoDB. Se cargaran datos de referencia.")
    seeds = [42, 100, 202, 333, 505, 777, 888, 999, 123, 456]
    sample_m = [23, 17, 19, 22, 15, 19, 19, 25, 19, 15]
    sample_sigma = [4.7808, 4.1519, 4.1556, 4.8622, 3.6952, 3.9517, 4.5176, 5.0000, 4.6934, 3.9768]
    sample_lambda = [0.01371, 0.01367, 0.00680, 0.00370, 0.06301, 0.02721, 0.01258, 0.00220, 0.00486, 0.02506]
    sample_mse = [0.22132, 0.24594, 0.20752, 0.21911, 0.26418, 0.22777, 0.22439, 0.20897, 0.21099, 0.26166]
    
    runs = []
    for i, s in enumerate(seeds):
        runs.append({
            "seed": s,
            "params": {"M": sample_m[i], "sigma": sample_sigma[i], "lambda": sample_lambda[i]},
            "metrics": {"mse": sample_mse[i]},
            "convergence": (np.geomspace(1.5, sample_mse[i], 30)).tolist()
        })
    print(f"Se cargaron {len(runs)} corridas de prueba exitosamente.")

## 1. Tabla de Resultados por Corrida

In [ ]:
records = []
for i, r in enumerate(runs, 1):
    records.append({
        "Corrida": i,
        "Semilla": r.get("seed"),
        "M (Centros)": r.get("params", {}).get("M"),
        "Sigma": r.get("params", {}).get("sigma"),
        "Lambda": r.get("params", {}).get("lambda"),
        "MSE Test": r.get("metrics", {}).get("mse")
    })

df_results = pd.DataFrame(records)
df_results

## 2. Resumen Estadistico Descriptivo

In [ ]:
mse_series = df_results["MSE Test"]

stats_summary = {
    "Metrica": ["Media", "Desviacion Estandar", "Mediana", "Minimo (Mejor)", "Maximo (Peor)", "Rango Intercuartilico (IQR)"],
    "Valor": [
        mse_series.mean(),
        mse_series.std(),
        mse_series.median(),
        mse_series.min(),
        mse_series.max(),
        mse_series.quantile(0.75) - mse_series.quantile(0.25)
    ]
}

df_stats = pd.DataFrame(stats_summary)
df_stats.style.format({"Valor": "{:.5f}"})

## 3. Curvas de Convergencia del Algoritmo SSA

In [ ]:
plt.figure(figsize=(10, 6))

all_convergences = []
for i, r in enumerate(runs, 1):
    conv = r.get("convergence", [])
    if conv:
        all_convergences.append(conv)
        plt.plot(conv, alpha=0.35, label=f"Corrida {i} (S:{r.get('seed')})")

if all_convergences:
    mean_conv = np.mean(all_convergences, axis=0)
    plt.plot(mean_conv, 'k--', linewidth=2.5, label="Promedio Global")

plt.title("Curvas de Convergencia de SSA en la Optimizacion de RBF (10 Corridas)", fontsize=13, fontweight='bold')
plt.xlabel("Iteraciones", fontsize=11)
plt.ylabel("Fitness (MSE de Validacion)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Distribucion del Error (Boxplot e Histograma)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Boxplot (con sintaxis actualizada de Matplotlib)
ax1.boxplot(mse_series, orientation='vertical', patch_artist=True, boxprops=dict(facecolor="lightblue"))
ax1.set_title("Diagrama de Caja (Boxplot) del MSE", fontsize=12, fontweight='bold')
ax1.set_ylabel("MSE en Test")
ax1.grid(True, linestyle="--", alpha=0.5)

# Histograma
ax2.hist(mse_series, bins=5, color="salmon", edgecolor="black", alpha=0.7)
ax2.set_title("Histograma de Distribucion del MSE", fontsize=12, fontweight='bold')
ax2.set_xlabel("MSE en Test")
ax2.set_ylabel("Frecuencia")
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()